In [1]:
# Data Preperation
import pandas as pd

df = pd.read_csv('../Data/news.tsv',sep='\t')
df.head()

,News ID,Category,Topic,Headline,News body,Title entity,Entity content
0,N10000,sports,soccer,Predicting Atlanta United's lineup against Col...,"Only FIVE internationals allowed, count em, FI...","{""Atlanta United's"": 'Atlanta United FC'}","{'Atlanta United FC': {'type': 'item', 'id': '..."
1,N10001,news,newspolitics,Mitch McConnell: DC statehood push is 'full bo...,WASHINGTON -- Senate Majority Leader Mitch McC...,"{'DC': 'Washington, D.C.'}","{'Washington, D.C.': {'type': 'item', 'id': 'Q..."
2,N10002,news,newsus,Home In North Highlands Damaged By Fire,NORTH HIGHLANDS (CBS13) Fire damaged a home ...,{},{}
3,N10003,news,newspolitics,Meghan McCain blames 'liberal media' and 'thir...,Meghan McCain is speaking out after a journali...,{},{}
4,N10004,news,newsworld,Today in History: Aug 1,"1714: George I becomes King Georg Ludwig, Elec...",{},{}


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113762 entries, 0 to 113761
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   News ID         113762 non-null  object
 1   Category        113762 non-null  object
 2   Topic           113762 non-null  object
 3   Headline        113762 non-null  object
 4   News body       113704 non-null  object
 5   Title entity    113762 non-null  object
 6   Entity content  113762 non-null  object
dtypes: object(7)
memory usage: 6.1+ MB


In [3]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /teamspace/studios/this_studio/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
import re
import string
import pandas as pd
from nltk.corpus import stopwords
import nltk
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from transformers import Trainer, TrainingArguments

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
2026-03-22 11:55:04.496925: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-poin

In [5]:
STOPWORDS = set(stopwords.words('english'))

def clean_text(text):
    
    if not isinstance(text,str):
        return ""

    text =  re.sub(r"<.*?>"," ",text)
    
    text = re.sub(r"http\S+|www\S+|https\S+"," ",text)

    text = re.sub(
        "["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F6FF"
        u"\U0001F1E0-\U0001F1FF"
        "]+",
        "",
        text
    )
    
    text = re.sub(r"[^a-zA-Z0-9\s.,!?]", " ", text)

    allowed = set(string.ascii_letters + string.digits + " .,!?")
    text = "".join(ch for ch in text if ch in allowed)

    text = text.lower()

    text = " ".join(text.split())

    return text


In [6]:
df["text"] = df["Headline"].fillna("") + " " + df["News body"].fillna("")
df = df.rename(columns={"Category": "label"}).dropna()
df["cleaned_text"] = df["text"].apply(clean_text)

In [7]:
from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()
df["label"] = le.fit_transform(df["label"])
num_classes = df["label"].nunique()

id2label = {i: label for i, label in enumerate(le.classes_)}
label2id = {label: i for i, label in id2label.items()}

print("\nLabel Mapping:")
for k, v in id2label.items():
    print(k, "->", v)


Label Mapping:
0 -> adexperience
1 -> autos
2 -> entertainment
3 -> europe
4 -> finance
5 -> foodanddrink
6 -> health
7 -> kids
8 -> lifestyle
9 -> movies
10 -> music
11 -> news
12 -> northamerica
13 -> sports
14 -> travel
15 -> tv
16 -> video
17 -> weather


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["cleaned_text"].values, df["label"].values, test_size=0.2, random_state=42)

In [9]:
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [10]:
# TRAINING SETUP
import torch
from transformers import get_linear_schedule_with_warmup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

EPOCHS = 3
BATCH_SIZE = 8
MAX_LEN = 96

Device: cuda


Bert-base-uncased

In [11]:
from transformers import BertTokenizer, BertForSequenceClassification
tokenizer_bert = BertTokenizer.from_pretrained("bert-base-uncased")

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [12]:
train_dataset = NewsDataset(X_train, y_train, tokenizer_bert, MAX_LEN)
test_dataset  = NewsDataset(X_test, y_test, tokenizer_bert, MAX_LEN)

In [13]:
model_bert = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
from transformers import (
    Trainer,
    TrainingArguments
)

In [15]:
training_args = TrainingArguments(
    output_dir="./PT1/Models/bert_output",

    evaluation_strategy="epoch",
    save_strategy="epoch",

    save_total_limit=1,   

    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,       
    per_device_eval_batch_size=BATCH_SIZE,

    gradient_accumulation_steps=2,         

    warmup_ratio=0.05,                   
    weight_decay=0.01,

    fp16=True,                             

    dataloader_num_workers=2,             
    remove_unused_columns=True,

    logging_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    report_to="none"
)

In [16]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=1)
    labels = eval_pred.label_ids

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
        "precision": precision_score(labels, preds, average="weighted"),
        "recall": recall_score(labels, preds, average="weighted")
    }

In [17]:
trainer = Trainer(
    model=model_bert,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
0,0.700200,0.694189,0.773361,0.770047,0.775454,0.773361
2,0.317400,0.744453,0.797238,0.796418,0.796384,0.797238


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"

TrainOutput(global_step=17055, training_loss=0.5571030793739392, metrics={'train_runtime': 2910.8932, 'train_samples_per_second': 93.748, 'train_steps_per_second': 5.859, 'total_flos': 1.346351767233408e+16, 'train_loss': 0.5571030793739392, 'epoch': 3.0})

DistilBert

In [18]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
tokenizer_distil = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [19]:
train_dataset_distil = NewsDataset(X_train, y_train, tokenizer_distil, MAX_LEN)
test_dataset_distil  = NewsDataset(X_test, y_test, tokenizer_distil, MAX_LEN)

In [20]:
model_distil = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [21]:
training_args_distil = TrainingArguments(

    output_dir="./PT1/Models/Distilbert_output",

    evaluation_strategy="epoch",
    save_strategy="epoch",

    save_total_limit=1,

    num_train_epochs=EPOCHS,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,

    warmup_ratio=0.05,
    weight_decay=0.01,

    fp16=True,

    dataloader_num_workers=2,
    remove_unused_columns=True,

    logging_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    report_to="none"
)

In [22]:
trainer_distil = Trainer(
    model=model_distil,
    args=training_args_distil,
    train_dataset=train_dataset_distil,
    eval_dataset=test_dataset_distil,

    compute_metrics=compute_metrics
)

trainer_distil.train()

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.698200,0.670915,0.778330,0.772908,0.772919,0.778330
2,0.523300,0.639220,0.792753,0.791799,0.793002,0.792753
3,0.344200,0.691385,0.792621,0.791815,0.792055,0.792621


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"

TrainOutput(global_step=8529, training_loss=0.5717033419104223, metrics={'train_runtime': 886.7693, 'train_samples_per_second': 307.734, 'train_steps_per_second': 9.618, 'total_flos': 6779851983692928.0, 'train_loss': 0.5717033419104223, 'epoch': 3.0})

In [23]:
import pandas as pd
import os

results = []

# Evaluate BERT
bert_eval = trainer.evaluate()

results.append({
    "model_name": "BERT-base-uncased",
    "accuracy": bert_eval.get("eval_accuracy"),
    "precision": bert_eval.get("eval_precision"),
    "recall": bert_eval.get("eval_recall"),
    "f1": bert_eval.get("eval_f1")
})

# Evaluate DistilBERT
distil_eval = trainer_distil.evaluate()

results.append({
    "model_name": "DistilBERT-base-uncased",
    "accuracy": distil_eval.get("eval_accuracy"),
    "precision": distil_eval.get("eval_precision"),
    "recall": distil_eval.get("eval_recall"),
    "f1": distil_eval.get("eval_f1")
})

results_df = pd.DataFrame(results)

print("\nTransformer Model Comparison:")
print(results_df)

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



Transformer Model Comparison:
                model_name  accuracy  precision    recall        f1
0        BERT-base-uncased  0.793281   0.794888  0.793281  0.793085
1  DistilBERT-base-uncased  0.792753   0.793002  0.792753  0.791799


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [24]:
csv_path = "./artifacts/model_comparisons.csv"

if os.path.exists(csv_path):
    results_df.to_csv(csv_path, mode="a", index=False, header=False)
else:
    results_df.to_csv(csv_path, index=False)

print("Transformer results appended to CSV")

Transformer results appended to CSV


In [25]:
best_row = results_df.sort_values(by="f1", ascending=False).iloc[0]

best_model_name = best_row["model_name"]

print("\nBest Transformer Model:", best_model_name)
print(best_row)


Best Transformer Model: BERT-base-uncased
model_name    BERT-base-uncased
accuracy               0.793281
precision              0.794888
recall                 0.793281
f1                     0.793085
Name: 0, dtype: object


In [26]:
import joblib

save_path = "./artifacts/PT1/Models"

if best_model_name == "BERT-base-uncased":
    model_bert.save_pretrained(save_path)
    tokenizer_bert.save_pretrained(save_path)

elif best_model_name == "DistilBERT-base-uncased":
    model_distil.save_pretrained(save_path)
    tokenizer_distil.save_pretrained(save_path)

# SAVE LABEL ENCODER
joblib.dump(le, "./artifacts/label_encoder.pkl")

print("Model + Tokenizer + LabelEncoder saved")

Model + Tokenizer + LabelEncoder saved


In [27]:
import pandas as pd

csv_path = "./artifacts/model_comparisons.csv"


df = pd.read_csv(csv_path)


df = df.sort_values(by="f1", ascending=False)


df.to_csv(csv_path, index=False)

print("Comparison of all Models based on F1 score")
print(df)

Comparison of all Models based on F1 score
                model_name  accuracy  precision    recall        f1
6        BERT-base-uncased  0.793281   0.794888  0.793281  0.793085
7  DistilBERT-base-uncased  0.792753   0.793002  0.792753  0.791799
0                 LR_TFIDF  0.781056   0.781056  0.781056  0.778427
1                SVM_TFIDF  0.775296   0.771919  0.775296  0.772787
2                  SVM_BOW  0.738358   0.743252  0.738358  0.739222
3                   LR_BOW  0.727101   0.729749  0.727101  0.727977
4             BiLSTM_GloVe  0.777802   0.614836  0.613629  0.611412
5            TextCNN_GloVe  0.747065   0.607571  0.550144  0.564745
